# M7 — VGG16 multiclasses + oversampling + Data Augmentation

**Objectif :** comparaison contrôlée avec M4. Même split, même VGG16 gelé, même tête multiclasses et mêmes paramètres d'entraînement. Seuls l'oversampling du **train** et la DA du **train** changent. **Aucune utilisation du test.**

Exécuter les cellules dans l'ordre. Vérifier les assertions et les effectifs affichés avant `fit()`. Les noms de classes sont lus dans les CSV, sans hypothèse sur leur orthographe.

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path
from imblearn.over_sampling import RandomOverSampler
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

tf.keras.utils.set_random_seed(42)
train_df = pd.read_csv("../covid/splits/train.csv")
val_df = pd.read_csv("../covid/splits/validation.csv")
required = {"filepath", "class"}
assert required.issubset(train_df.columns) and required.issubset(val_df.columns), (
    "Les CSV doivent contenir 'filepath' et 'class' (étiquettes originales à 4 classes)."
)
assert train_df["class"].notna().all() and val_df["class"].notna().all()
assert train_df["filepath"].notna().all() and val_df["filepath"].notna().all()
classes = sorted(train_df["class"].astype(str).unique().tolist())
assert len(classes) == 4, f"Quatre classes attendues : {classes}"
assert set(val_df["class"].astype(str)) == set(classes)
assert len(train_df) == 13545 and len(val_df) == 3387, (
    f"Split inattendu : train={len(train_df)}, validation={len(val_df)}"
)
assert train_df["filepath"].is_unique and val_df["filepath"].is_unique
assert set(train_df["filepath"]).isdisjoint(set(val_df["filepath"])), "Fuite train/validation"
print("Classes (ordre fixe) :", classes)
print("Train avant OS :\n", train_df["class"].value_counts())
print("Validation inchangée :\n", val_df["class"].value_counts())

Classes (ordre fixe) : ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
Train avant OS :
 class
Normal             6495
Lung_Opacity       3895
COVID              2314
Viral Pneumonia     841
Name: count, dtype: int64
Validation inchangée :
 class
Normal             1607
Lung_Opacity        977
COVID               579
Viral Pneumonia     224
Name: count, dtype: int64


## 1. Oversampling : train uniquement

`RandomOverSampler(random_state=42)` réplique les observations minoritaires jusqu'à l'effectif de la classe majoritaire (stratégie `auto` en multiclasses). Contrairement au binaire E7, **quatre classes** sont équilibrées ici.

In [2]:
ros = RandomOverSampler(random_state=42)
train_os_df, _ = ros.fit_resample(train_df, train_df["class"].astype(str))
train_os_df = train_os_df.sample(frac=1, random_state=42).reset_index(drop=True)
assert len(train_os_df["class"].value_counts().unique()) == 1
assert len(val_df) == 3387
assert set(train_os_df["filepath"]).issubset(set(train_df["filepath"]))
print("Train après OS :\n", train_os_df["class"].value_counts())
print("Total train OS :", len(train_os_df))
print("Validation non rééchantillonnée :", len(val_df))

Train après OS :
 class
Lung_Opacity       6495
Normal             6495
COVID              6495
Viral Pneumonia    6495
Name: count, dtype: int64
Total train OS : 25980
Validation non rééchantillonnée : 3387


## 2. Générateurs

DA médicale harmonisée : rotation ±10°, translations horizontale et verticale 10 %, zoom 10 %, **sans flip ni shear**. Prétraitement VGG16 sur train et validation. `shuffle=False` pour la validation afin d'aligner prédictions et étiquettes.

In [3]:
train_gen_df = train_os_df.copy()
val_gen_df = val_df.copy()
train_gen_df["class"] = train_gen_df["class"].astype(str)
val_gen_df["class"] = val_gen_df["class"].astype(str)

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_dataframe(
    train_gen_df, x_col="filepath", y_col="class", classes=classes,
    target_size=(224, 224), color_mode="rgb", class_mode="categorical",
    batch_size=32, shuffle=True, seed=42
)
val_generator = val_datagen.flow_from_dataframe(
    val_gen_df, x_col="filepath", y_col="class", classes=classes,
    target_size=(224, 224), color_mode="rgb", class_mode="categorical",
    batch_size=32, shuffle=False
)
assert train_generator.class_indices == val_generator.class_indices
assert len(train_generator.class_indices) == 4
assert train_generator.samples == len(train_os_df)
assert val_generator.samples == len(val_df)
print("Mapping :", train_generator.class_indices)

Found 25980 validated image filenames belonging to 4 classes.
Found 3387 validated image filenames belonging to 4 classes.
Mapping : {'COVID': 0, 'Lung_Opacity': 1, 'Normal': 2, 'Viral Pneumonia': 3}


## 3. Modèle et entraînement

Architecture identique à M4 : VGG16 ImageNet gelé, GAP, Dense(128, ReLU), Dropout(0,5), Dense(4, softmax). Adam 0,001 ; 5 epochs maximum ; EarlyStopping patience 2 ; checkpoint sur `val_loss`.

In [4]:
base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False
model_m7 = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(4, activation="softmax")
])
model_m7.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model_m7.summary()
assert len(model_m7.trainable_weights) > 0
assert not base_model.trainable

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)                   │ (None, 7, 7, 512)           │      14,714,688 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 512)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │          65,664 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 4)                   │             516 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 14,780,868 (56.38 MB)

 Trainable params: 66,180 (258.52 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
    ModelCheckpoint("M7_VGG16_multiclass_OS_DA_best.keras", monitor="val_loss", save_best_only=True)
]
history_m7 = model_m7.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/5
 27/812 ━━━━━━━━━━━━━━━━━━━━ 1:31:14 7s/step - accuracy: 0.4606 - loss: 2.0907 

## 4. Évaluation — validation uniquement

F1 et recall **macro** (comparaison des quatre classes) ; rappel de la classe COVID (enjeu médical). Matrice de confusion avec ordre des classes explicite. Aucun seuil binaire 0,5 : en multiclasses, prédiction par `argmax`.

In [ ]:
history_df = pd.DataFrame(history_m7.history)
history_df.index = np.arange(1, len(history_df) + 1)
history_df.index.name = "epoch"
history_df.to_csv("M7_history.csv")
display(history_df)

val_generator.reset()
y_prob_m7 = model_m7.predict(val_generator, verbose=1)
y_true_m7 = val_generator.classes
y_pred_m7 = np.argmax(y_prob_m7, axis=1)
assert len(y_pred_m7) == len(y_true_m7) == len(val_df)
assert y_prob_m7.shape[1] == 4
report_m7 = classification_report(
    y_true_m7, y_pred_m7, labels=list(range(4)),
    target_names=classes, digits=4, output_dict=True, zero_division=0
)
print(classification_report(
    y_true_m7, y_pred_m7, labels=list(range(4)),
    target_names=classes, digits=4, zero_division=0
))
pd.DataFrame(report_m7).transpose().to_csv("M7_classification_report.csv")
print("F1 macro :", round(report_m7["macro avg"]["f1-score"], 4))
print("Recall macro :", round(report_m7["macro avg"]["recall"], 4))
covid_label = next((name for name in classes if name.strip().upper() == "COVID"), None)
if covid_label is not None:
    print("Recall COVID :", round(report_m7[covid_label]["recall"], 4))
else:
    print("Vérifier le libellé COVID dans la liste des classes :", classes)

cm_m7 = confusion_matrix(y_true_m7, y_pred_m7, labels=list(range(4)))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm_m7, display_labels=classes).plot(
    ax=ax, cmap="Blues", values_format="d", xticks_rotation=35, colorbar=False
)
ax.set_title("M7 — Matrice de confusion (validation)")
fig.tight_layout()
fig.savefig("M7_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
for metric in ["loss", "accuracy"]:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(history_m7.history[metric], label="Train")
    ax.plot(history_m7.history[f"val_{metric}"], label="Validation")
    ax.set(title=f"M7 — {metric}", xlabel="Epoch", ylabel=metric)
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(f"M7_{metric}.png", dpi=150, bbox_inches="tight")
    plt.show()

### À conserver pour le rapport

- `M7_VGG16_multiclass_OS_DA_best.keras` (meilleur modèle selon `val_loss`)
- `M7_history.csv`, `M7_classification_report.csv`
- `M7_confusion_matrix.png`, `M7_loss.png`, `M7_accuracy.png`

Comparer M4/M7 **sur validation** avec F1 macro, recall macro, recall COVID et matrices de confusion. Ne pas utiliser le test pour choisir le modèle.